## Data Improvement

In [110]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np

In [111]:
df = pd.read_csv('df_corrupted.csv')


In [112]:
import pandas as pd
import re
from datetime import datetime

def clean_datetime(dt_str):
    """Convertit différents formats de date en format standard YYYY-MM-DD HH:MM:SS"""
    if pd.isna(dt_str):
        return None
    
    dt_str = str(dt_str).strip()
    
    # Format: "11-01-2023 20 heure 24 minutes"
    match = re.match(r'(\d{1,2})-(\d{1,2})-(\d{4})\s+(\d{1,2})\s+heure\s+(\d{1,2})\s+minutes?', dt_str)
    if match:
        day, month, year, hour, minute = match.groups()
        return f"{year}-{month.zfill(2)}-{day.zfill(2)} {hour.zfill(2)}:{minute.zfill(2)}:00"
    
    # Format: "6 janvier 2023 à 9h08"
    months_fr = {
        'janvier': '01', 'février': '02', 'mars': '03', 'avril': '04',
        'mai': '05', 'juin': '06', 'juillet': '07', 'août': '08',
        'septembre': '09', 'octobre': '10', 'novembre': '11', 'décembre': '12'
    }
    match = re.match(r'(\d{1,2})\s+(\w+)\s+(\d{4})\s+à\s+(\d{1,2})h(\d{2})', dt_str)
    if match:
        day, month_name, year, hour, minute = match.groups()
        month = months_fr.get(month_name.lower(), '01')
        return f"{year}-{month}-{day.zfill(2)} {hour.zfill(2)}:{minute}:00"
    
    # Format: "19-01-2023 21h07"
    match = re.match(r'(\d{1,2})-(\d{1,2})-(\d{4})\s+(\d{1,2})h(\d{2})', dt_str)
    if match:
        day, month, year, hour, minute = match.groups()
        return f"{year}-{month.zfill(2)}-{day.zfill(2)} {hour.zfill(2)}:{minute}:00"
    
    # Format déjà correct: "2023-01-10 17:26:00"
    if re.match(r'\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}', dt_str):
        return dt_str
    
    return None

def clean_distance(dist_str):
    """Extrait la valeur numérique de la distance"""
    if pd.isna(dist_str):
        return None
    
    # Extraire uniquement les chiffres et le point décimal
    match = re.search(r'(\d+\.?\d*)', str(dist_str))
    if match:
        return float(match.group(1))
    return None

def clean_fare(fare_str):
    """Extrait la valeur numérique du montant"""
    if pd.isna(fare_str):
        return None
    
    # Supprimer tous les symboles de devise et espaces, garder chiffres et point
    cleaned = re.sub(r'[£€¥$\s]', '', str(fare_str))
    match = re.search(r'(\d+\.?\d*)', cleaned)
    if match:
        return float(match.group(1))
    return None

# Charger le dataset

# Appliquer les corrections
df['pickup_datetime'] = df['pickup_datetime'].apply(clean_datetime)
df['dropoff_datetime'] = df['dropoff_datetime'].apply(clean_datetime)
df['trip_distance_miles'] = df['trip_distance_miles'].apply(clean_distance)
df['fare_amount'] = df['fare_amount'].apply(clean_fare)


### Suppression des doublons

In [113]:
df = df.drop_duplicates()
len(df)

800

### imputation des distances null et negatifs

In [114]:
from sklearn.linear_model import LinearRegression


# 2. Entraîner sur données valides
df_train = pd.read_csv("Taxi.csv")
df_train = df_train[
    (df_train['fare_amount'] > 0) & (df_train['fare_amount'] <= 150) &
    (df_train['trip_distance_miles'] > 0) & (df_train['trip_distance_miles'] <= 50)
]

model = LinearRegression()
model.fit(df_train[['fare_amount']], df_train['trip_distance_miles'])

# 3. Identifier les lignes à imputer
to_impute = (
    ((df['trip_distance_miles'].isna()) | (df['trip_distance_miles'] < 0)) &
    (df['fare_amount'] > 0) & (df['fare_amount'] <= 150)
)

# 4. Imputer
if to_impute.sum() > 0:
    df.loc[to_impute, 'trip_distance_miles'] = model.predict(df.loc[to_impute, ['fare_amount']])
else:
    print("Aucune ligne à imputer")



### Correction des coordonnées gps erronées

In [115]:

# 1) Charger la limite de NYC (GeoJSON depuis ArcGIS)
nyc_url = ("https://services5.arcgis.com/GfwWNkhOj9bNBqoJ"
           "/arcgis/rest/services/NYC_Borough_Boundary_Water_Included"
           "/FeatureServer/0/query?where=1=1&outFields=*&f=geojson")
nyc = gpd.read_file(nyc_url)

# 2) Charger les polygones d'eau (shapefile téléchargé)
water = gpd.read_file("tl_2023_36061_areawater.shp")

# 3) Mettre les deux jeux de données dans la même projection (lat/lon EPSG:4326)
nyc = nyc.to_crs(epsg=4326)
water = water.to_crs(epsg=4326)

NYC_BBOX = {
    'lat_min': 40.4,
    'lat_max': 41.0,
    'lon_min': -74.4,
    'lon_max': -73.6
}

def check_location(lat, lon):
    pt = Point(lon, lat)
    
    # Vérifier l'eau en premier
    if water.contains(pt).any():
        return "water"
    
    # Si pas dans l'eau, vérifier NYC
    if nyc.contains(pt).any():
        return "NYC"
    
    if (NYC_BBOX['lat_min'] <= lat <= NYC_BBOX['lat_max'] and
        NYC_BBOX['lon_min'] <= lon <= NYC_BBOX['lon_max']):
        return True
    
    # Si ni eau ni NYC
    return "Hors NYC"

In [116]:
import numpy as np

def is_distance_erroneous(row):
    """Vérifie si la distance est erronée selon les critères"""
    distance = row['trip_distance_miles']
    fare = row['fare_amount']

    # Distance négative ou supérieure à 50
    if distance < 0 or distance > 50 or pd.isna(distance):
        return True
    
    return False

# Centre approximatif de Manhattan
NYC_CENTER = {'lat': 40.7589, 'lon': -73.9851}

def calculate_bearing(lat1, lon1, lat2, lon2):
    """Calcule l'angle (bearing) entre deux points en degrés"""
    lat1_rad = np.radians(lat1)
    lat2_rad = np.radians(lat2)
    dlon = np.radians(lon2 - lon1)
    
    x = np.sin(dlon) * np.cos(lat2_rad)
    y = np.cos(lat1_rad) * np.sin(lat2_rad) - np.sin(lat1_rad) * np.cos(lat2_rad) * np.cos(dlon)
    
    bearing = np.arctan2(x, y)
    return np.degrees(bearing)

def move_point(lat, lon, distance_miles, bearing_degrees):
    """Déplace un point selon une distance et un bearing"""
    R = 3958.8  # Rayon de la Terre en miles
    
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    bearing_rad = np.radians(bearing_degrees)
    
    new_lat_rad = np.arcsin(
        np.sin(lat_rad) * np.cos(distance_miles / R) +
        np.cos(lat_rad) * np.sin(distance_miles / R) * np.cos(bearing_rad)
    )
    
    new_lon_rad = lon_rad + np.arctan2(
        np.sin(bearing_rad) * np.sin(distance_miles / R) * np.cos(lat_rad),
        np.cos(distance_miles / R) - np.sin(lat_rad) * np.sin(new_lat_rad)
    )
    
    return np.degrees(new_lat_rad), np.degrees(new_lon_rad)

def find_valid_point_in_nyc(ref_lat, ref_lon, distance, original_lat=None, original_lon=None):
    """Trouve un point valide dans NYC à une distance donnée d'un point de référence"""
    
    # Stratégie 1: Utiliser la direction originale si disponible
    if original_lat is not None and original_lon is not None:
        bearing = calculate_bearing(ref_lat, ref_lon, original_lat, original_lon)
        new_lat, new_lon = move_point(ref_lat, ref_lon, distance, bearing)
        
        if check_location(new_lat, new_lon) == "NYC":
            return new_lat, new_lon
    
    # Stratégie 2: Essayer la direction vers le centre de NYC
    bearing_to_center = calculate_bearing(ref_lat, ref_lon, NYC_CENTER['lat'], NYC_CENTER['lon'])
    new_lat, new_lon = move_point(ref_lat, ref_lon, distance, bearing_to_center)
    
    if check_location(new_lat, new_lon) == "NYC":
        return new_lat, new_lon
    
    # Stratégie 3: Essayer plusieurs directions (8 directions cardinales)
    for bearing in [0, 45, 90, 135, 180, 225, 270, 315]:
        new_lat, new_lon = move_point(ref_lat, ref_lon, distance, bearing)
        
        if check_location(new_lat, new_lon) == "NYC":
            return new_lat, new_lon
    
    # Aucune direction ne fonctionne
    return None, None

def correct_points(df):
    """Corrige les points pickup/dropoff erronés"""
    df = df.copy()
    
    # Ne considérer que les lignes avec distance correcte
    valid_distance_mask = ~df.apply(is_distance_erroneous, axis=1)
    
    print(f"Lignes avec distance correcte : {valid_distance_mask.sum()}")
    
    corrections_made = 0
    
    for idx in df[valid_distance_mask].index:
        row = df.loc[idx]
        
        pickup_lat = row['pickup_latitude']
        pickup_lon = row['pickup_longitude']
        dropoff_lat = row['dropoff_latitude']
        dropoff_lon = row['dropoff_longitude']
        distance = row['trip_distance_miles']
        
        pickup_status = check_location(pickup_lat, pickup_lon)
        dropoff_status = check_location(dropoff_lat, dropoff_lon)
        
        # Cas 1: Pickup correct, dropoff erroné
        if pickup_status == "NYC" and dropoff_status != "NYC":
            new_lat, new_lon = find_valid_point_in_nyc(
                pickup_lat, pickup_lon, distance, 
                dropoff_lat, dropoff_lon
            )
            
            if new_lat is not None:
                df.loc[idx, 'dropoff_latitude'] = new_lat
                df.loc[idx, 'dropoff_longitude'] = new_lon
                corrections_made += 1
        
        # Cas 2: Dropoff correct, pickup erroné
        elif dropoff_status == "NYC" and pickup_status != "NYC":
            new_lat, new_lon = find_valid_point_in_nyc(
                dropoff_lat, dropoff_lon, distance,
                pickup_lat, pickup_lon
            )
            
            if new_lat is not None:
                df.loc[idx, 'pickup_latitude'] = new_lat
                df.loc[idx, 'pickup_longitude'] = new_lon
                corrections_made += 1
    
    print(f"Points corrigés : {corrections_made}")
    
    return df

# Utilisation:
df = correct_points(df)

Lignes avec distance correcte : 799
Points corrigés : 177


### Correction des valeurs de la colonne fair_amount

#### 1 fair_amount

In [117]:
from sklearn.linear_model import LinearRegression
import pandas as pd


# 2. Entraîner sur données valides
df_train = pd.read_csv("Taxi.csv")
df_train = df_train[
    (df_train['fare_amount'] > 0) & (df_train['fare_amount'] <= 150) &
    (df_train['trip_distance_miles'] > 0) & (df_train['trip_distance_miles'] <= 50)
]

model = LinearRegression()
model.fit(df_train[['trip_distance_miles']], df_train['fare_amount'])

print(f"Équation : fare = {model.coef_[0]:.2f} × distance + {model.intercept_:.2f}")
print(f"R² = {model.score(df_train[['trip_distance_miles']], df_train['fare_amount']):.3f}\n")

# 3. Identifier les lignes à imputer
to_impute = (
    ((df['fare_amount'].isna()) | (df['fare_amount'] < 0)) &
    (df['trip_distance_miles'] > 0) & (df['trip_distance_miles'] <= 50)
)

# 4. Imputer
if to_impute.sum() > 0:
    df.loc[to_impute, 'fare_amount'] = model.predict(df.loc[to_impute, ['trip_distance_miles']])
else:
    print("Aucune ligne à imputer")



Équation : fare = 2.92 × distance + 5.16
R² = 0.658



### Correction des incoherences temporelles

In [118]:
# Conversion en objets datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'], errors='coerce')

# Masque d'erreur : dropoff avant pickup OU valeurs nulles (mais pas les deux nulles)
mask_error = (
    (df['dropoff_datetime'] < df['pickup_datetime']) | 
    ((df['dropoff_datetime'].isna() | df['pickup_datetime'].isna()) & 
     ~(df['dropoff_datetime'].isna() & df['pickup_datetime'].isna()))
)

print("--- Lignes en erreur AVANT Correction ---")
if mask_error.any():
    print(df.loc[mask_error, ['pickup_datetime', 'dropoff_datetime', 'trip_distance_miles']])
else:
    print("Aucune erreur détectée.")

# Calcul vitesse moyenne de référence (données saines)
mask_sain = (~mask_error) & (df['trip_distance_miles'] > 0)
df_sain = df.loc[mask_sain].copy()

df_sain['duration_h'] = (df_sain['dropoff_datetime'] - df_sain['pickup_datetime']).dt.total_seconds() / 3600
df_sain['speed_mph'] = df_sain['trip_distance_miles'] / df_sain['duration_h']

# Référentiel : Heure -> Vitesse Médiane
ref_speed = df_sain.groupby(df_sain['dropoff_datetime'].dt.hour)['speed_mph'].median()

# Correction des erreurs
if mask_error.any():
    for idx in df[mask_error].index:
        pickup = df.at[idx, 'pickup_datetime']
        dropoff = df.at[idx, 'dropoff_datetime']
        distance = df.at[idx, 'trip_distance_miles']
        
        # Cas 1 : Dropoff existe mais pas pickup
        if pd.notna(dropoff) and pd.isna(pickup) and distance > 0:
            hour = dropoff.hour
            speed = ref_speed.get(hour, 12.0) if ref_speed.get(hour, 0) > 0 else 12.0
            duration_seconds = (distance / speed) * 3600
            df.at[idx, 'pickup_datetime'] = dropoff - pd.Timedelta(seconds=duration_seconds)
        
        # Cas 2 : Pickup existe mais pas dropoff
        elif pd.notna(pickup) and pd.isna(dropoff) and distance > 0:
            hour = pickup.hour
            speed = ref_speed.get(hour, 12.0) if ref_speed.get(hour, 0) > 0 else 12.0
            duration_seconds = (distance / speed) * 3600
            df.at[idx, 'dropoff_datetime'] = pickup + pd.Timedelta(seconds=duration_seconds)
        
        # Cas 3 : Les deux existent mais dropoff < pickup
        elif pd.notna(pickup) and pd.notna(dropoff) and dropoff < pickup and distance > 0:
            hour = dropoff.hour
            speed = ref_speed.get(hour, 12.0) if ref_speed.get(hour, 0) > 0 else 12.0
            duration_seconds = (distance / speed) * 3600
            df.at[idx, 'pickup_datetime'] = dropoff - pd.Timedelta(seconds=duration_seconds)

print("\n--- Lignes en erreur APRÈS Correction ---")
if mask_error.any():
    print(df.loc[mask_error, ['pickup_datetime', 'dropoff_datetime', 'trip_distance_miles']])

# Sauvegarder le dataset nettoyé
print("\nNettoyage terminé!")


--- Lignes en erreur AVANT Correction ---
        pickup_datetime    dropoff_datetime  trip_distance_miles
0   2023-02-20 17:27:00                 NaT                 1.03
11  2023-02-07 13:55:00                 NaT                 2.30
17  2023-01-22 10:35:00                 NaT                 3.38
23  2023-01-25 12:20:00                 NaT                 2.86
34                  NaT 2023-01-10 08:40:00                 2.70
..                  ...                 ...                  ...
775 2023-01-08 22:53:00                 NaT                 2.23
780                 NaT 2023-02-24 19:38:00                 1.84
784                 NaT 2023-02-18 23:03:00                 3.62
797                 NaT 2023-03-01 14:22:00                 3.57
799 2023-01-02 14:03:00                 NaT                 2.70

[93 rows x 3 columns]

--- Lignes en erreur APRÈS Correction ---
                  pickup_datetime              dropoff_datetime  \
0   2023-02-20 17:27:00.000000000 2023-02-20 

## Correction des attributs catégorielles et discrets

In [119]:
#corriger les nan dans payment_type en les remplaçant par la valeur la plus fréquente
most_frequent_payment_type = df['payment_type'].mode()[0]
df['payment_type'].fillna(most_frequent_payment_type, inplace=True)

#corriger les nan dans passenger_count en les remplaçant par la valeur la plus fréquente
most_frequent_passenger_count = df['passenger_count'].mode()[0]
df['passenger_count'].fillna(most_frequent_passenger_count, inplace=True)

C:\Users\dell\AppData\Local\Temp\ipykernel_13868\3671236580.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['payment_type'].fillna(most_frequent_payment_type, inplace=True)
C:\Users\dell\AppData\Local\Temp\ipykernel_13868\3671236580.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

In [120]:
df.to_csv("Taxi_corrupted_corrigé.csv", index=False)